<h1> Objetivo Del Estudio </h1>

El objetivo es analizar una base de datos de libros para identificar patrones relevantes que permitan construir una propuesta de valor para una forma de lectura, considerando:

El Volumen de libros recientes
La Popularidad de ratings y reviews
Editoriales más activas
Autores mejor valorados
Comportamiento de usuarios

<h1> Exploracion De Datos </h1>

In [4]:
import pandas as pd
from sqlalchemy import create_engine

db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-final-project-db'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string)

# Función reutilizable para ejecutar consultas
def run_query(query):
    with engine.connect() as conn:
        return pd.read_sql(query, conn)


<h1> Consultas SQL</h1>

In [6]:
query_libros_modernos = """
SELECT COUNT(book_id) AS total_libros_modernos
FROM books
WHERE publication_date > '2000-01-01';
"""

resultado = run_query(query_libros_modernos)
print("Libros publicados después del 1 de enero de 2000:")
print(resultado)

Libros publicados después del 1 de enero de 2000:
   total_libros_modernos
0                    819


son 819 libros los que se publicaron despues del primer mes del año 2000 los cuales fueron los libros as modernos que se pudieron publicar en ese año y representan el 82% del catálogo total

<h1> Número de reseñas y calificación promedio por libro </h1>

In [9]:
query_resenas_calificaciones = """
SELECT
    b.book_id, 
    b.title,
    COUNT(DISTINCT rv.review_id) AS num_resenas,
    ROUND(AVG(rt.rating)::NUMERIC, 2) AS calif_promedio
FROM books b
LEFT JOIN reviews rv ON b.book_id = rv.book_id
LEFT JOIN ratings rt ON b.book_id = rt.book_id
GROUP BY b.book_id, b.title
ORDER BY calif_promedio DESC NULLS LAST;
"""

resultado_resenas = run_query(query_resenas_calificaciones)
print("Número de reseñas y calificación promedio por libro:")
print(resultado_resenas.head(10))

Número de reseñas y calificación promedio por libro:
   book_id                                              title  num_resenas  \
0       86      Arrows of the Queen (Heralds of Valdemar  #1)            2   
1      901  The Walking Dead  Book One (The Walking Dead #...            2   
2      390                                    Light in August            2   
3      972  Wherever You Go  There You Are: Mindfulness Me...            2   
4      136  Captivating: Unveiling the Mystery of a Woman'...            2   
5      610                           Tai-Pan (Asian Saga  #2)            2   
6      625  The Adventures of Tom Sawyer and Adventures of...            1   
7      297                                         Hard Times            2   
8       20              A Fistful of Charms (The Hollows  #4)            2   
9      347  In the Hand of the Goddess (Song of the Liones...            2   

   calif_promedio  
0             5.0  
1             5.0  
2             5.0  
3       

en este punto se puede analizar que salieron 10 libros los cuales obtuvieron las mejores calificaciones ya que cada uno obtuvo la calificacion mayor y fueron los mejores libros que se publicaron, sus numeros son excelentes y esto beneficia mucho el negocio

<h1> Editorial con más libros </h1>

In [11]:
query_editorial_mas_libros = """
SELECT
    p.publisher,
    COUNT(b.book_id) AS total_libros
FROM books b
JOIN publishers p ON b.publisher_id = p.publisher_id
WHERE b.num_pages > 50
GROUP BY p.publisher
ORDER BY total_libros DESC
LIMIT 1;
"""

resultado_editorial = run_query(query_editorial_mas_libros)
print("Editorial con más libros de más de 50 páginas:")
print(resultado_editorial)

Editorial con más libros de más de 50 páginas:
       publisher  total_libros
0  Penguin Books            42


en este editorial el libro penguin books fue el que mas paginas tuvo ya que fue el mas calificado por tener mas de 50 paginas que los demas libros escritos

<h1> Autor con mayor calificación </h1>

In [14]:
query_nombre = """
SELECT
    a.author,
    ROUND(AVG(rt.rating)::NUMERIC, 2) AS calif_promedio,
    COUNT(rt.rating_id) AS total_calif
FROM authors a
JOIN books b ON a.author_id = b.author_id
JOIN ratings rt ON b.book_id = rt.book_id
GROUP BY a.author
HAVING COUNT(rt.rating_id) >= 50
ORDER BY calif_promedio DESC
LIMIT 1;
"""

resultado = run_query(query_nombre)
print("Autor con mayor calificación promedio (mínimo 50 calificaciones):")
print(resultado)

Autor con mayor calificación promedio (mínimo 50 calificaciones):
           author  calif_promedio  total_calif
0  Diana Gabaldon             4.3           50


el autor diana gabaldon fue la que obtuvo la mayor calificacion en uno de sus libros ya que fue la elegida por ser la mejor escritora y esto ayudara mucho al negocio por que la venta de sus libros sera mucho mayor

<h1> Promedio de reseñas de usuarios activos </h1>

In [16]:
query_usuarios_activos = """
WITH usuarios_activos AS (
    SELECT username
    FROM ratings
    GROUP BY username
    HAVING COUNT(book_id) > 50
)
SELECT
    ROUND(AVG(num_resenas)::NUMERIC, 2) AS promedio_resenas
FROM (
    SELECT rv.username, COUNT(rv.review_id) AS num_resenas
    FROM reviews rv
    WHERE rv.username IN (SELECT username FROM usuarios_activos)
    GROUP BY rv.username
) sub;
"""

resultado_usuarios = run_query(query_usuarios_activos)
print("Promedio de reseñas de usuarios activos (más de 50 calificaciones):")
print(resultado_usuarios)

Promedio de reseñas de usuarios activos (más de 50 calificaciones):
   promedio_resenas
0             24.33


se obtuvo un promedio de 24.33 % de los usuarios que actualmente se encuentran activos

<h1> Conclusiones </h1>

Con este análisis puedes construir una propuesta sólida: catálogo moderno, libros destacados por reputación, alianza con la editorial líder, autor estrella para marketing y un sistema de gamificación para retener a los usuarios más activos.